In [37]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder, LabelEncoder, MinMaxScaler
from sklearn.compose import ColumnTransformer

df = pd.read_csv("./titanic_data_updated.csv")

In [38]:
df.sample(4)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
701,702,yes,first,"Silverthorne, Mr. Spencer Victor",male,35.0,0,0,PC 17475,26.2875,E24,S
514,515,no,third,"Coleff, Mr. Satio",male,24.0,0,0,349209,7.4958,NaN,S
14,15,no,third,"Vestrom, Miss. Hulda Amanda Adolfina",female,14.0,0,0,350406,7.8542,NaN,S
292,293,no,second,"Levy, Mr. Rene Jacques",male,36.0,0,0,SC/Paris 2163,12.8750,D,C


In [39]:
df.drop(columns=['Name','PassengerId','Ticket'],inplace=True)

In [40]:
df.sample(2)

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Cabin,Embarked
57,no,third,male,28.5,0,0,7.2292,NaN,C
427,yes,second,female,19.0,0,0,26.0000,NaN,S


In [41]:
df['FamilyMembers'] = df['Parch'] + df['SibSp'] + 1;
df.drop(columns=['Parch','SibSp'],inplace=True)

In [42]:
df.sample(2)

,Survived,Pclass,Sex,Age,Fare,Cabin,Embarked,FamilyMembers
654,no,third,female,18.0,6.75,NaN,Q,1
63,no,third,male,4.0,27.90,NaN,S,6


### Train Test Split

In [43]:
x = df.drop(columns='Survived') # features
y = df['Survived'] # Target

x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.2, random_state=42)

In [44]:
x_train.isnull().sum()

Pclass             0
Sex                0
Age              140
Fare               0
Cabin            553
Embarked           2
FamilyMembers      0
dtype: int64

In [45]:
x_test.isnull().sum()

Pclass             0
Sex                0
Age               37
Fare               0
Cabin            134
Embarked           0
FamilyMembers      0
dtype: int64

Imputation using column transformer

In [46]:
imputer_transformer = ColumnTransformer(
    transformers=[
        ('age',SimpleImputer(missing_values=np.nan, strategy='mean'),['Age']),
        ('embarked',SimpleImputer(missing_values=np.nan,strategy='most_frequent'),['Embarked']),
        ('Cabin',SimpleImputer(missing_values=np.nan,strategy='constant',fill_value='Missing',add_indicator=True),['Cabin'])
    ],
    remainder='passthrough',
    verbose_feature_names_out=False
)

imputer_transformer.set_output(transform='pandas')

imputer_transformer.fit(x_train)
x_train = imputer_transformer.transform(x_train)
x_test = imputer_transformer.transform(x_test)

In [47]:
x_train.isnull().sum()

Age                       0
Embarked                  0
Cabin                     0
missingindicator_Cabin    0
Pclass                    0
Sex                       0
Fare                      0
FamilyMembers             0
dtype: int64

In [48]:
x_test.isnull().sum()

Age                       0
Embarked                  0
Cabin                     0
missingindicator_Cabin    0
Pclass                    0
Sex                       0
Fare                      0
FamilyMembers             0
dtype: int64

### Outliers Detection and Handling

Age

In [49]:
mean_age = x_train['Age'].mean();
std_age = x_train['Age'].std();

x_train['zscore_age'] = (x_train['Age']-mean_age)/std_age;
outliers_age = x_train[abs(x_train['zscore_age'])>3]

print("Number of outliers: ",len(outliers_age))

x_train = x_train[abs(x_train['zscore_age'])<=3]

Number of outliers:  5


In [55]:
# x_train.sort_values(by='Age',ascending=0).head(5)
x_train

,Age,Embarked,Cabin,missingindicator_Cabin,Pclass,Sex,Fare,FamilyMembers,zscore_age
331,45.500000,S,C124,False,first,male,28.5000,1,1.231398e+00
733,23.000000,S,Missing,True,second,male,13.0000,1,-5.001304e-01
382,32.000000,S,Missing,True,third,male,7.9250,1,1.924808e-01
704,26.000000,S,Missing,True,third,male,7.8542,2,-2.692600e-01
813,6.000000,S,Missing,True,third,female,31.2750,7,-1.808396e+00
...,...,...,...,...,...,...,...,...,...
106,21.000000,S,Missing,True,third,female,7.6500,1,-6.540440e-01
270,29.498846,S,Missing,True,first,male,31.0000,1,2.734055e-16
860,41.000000,S,Missing,True,third,male,14.1083,3,8.850920e-01
435,14.000000,S,B96 B98,False,first,female,64.3625,4,-1.192742e+00


In [51]:
# deleting outliers age
x_train = x_train[abs(x_train['zscore_age'])<=3]

Fare

In [52]:
fare_q1 = x_train['Fare'].quantile(0.25)
fare_q3 = x_train['Fare'].quantile(0.75)

iqr = fare_q3 - fare_q1;

max_range = fare_q3 + (iqr*1.5);
min_range = max(0,fare_q1 - (iqr*1.5));

x_train['Fare'] = x_train['Fare'].clip(min_range,max_range)

In [54]:
print(x_train['Fare'].min())
print(x_train['Fare'].max())

0.0
64.3625


In [57]:
x_train.drop(columns='zscore_age',inplace=True)

Encoding and Scaling

In [59]:
# making a column 'Cabin_deck' by taking the first character of cabin name
x_train['Cabin_deck'] = x_train['Cabin'].astype(str).str[0];
x_test['Cabin_deck'] = x_test['Cabin'].astype(str).str[0];

In [60]:
encoder_scaler = ColumnTransformer(
    transformers=[
        ('pclass',OrdinalEncoder(categories=[['third','second','first']]),['Pclass']),
        ('sex_embarked_cabinDeck',OneHotEncoder(sparse_output=False,drop='first'),['Sex','Embarked','Cabin_deck']),
        ('age',StandardScaler(),['Age']),
        ('fare_family',MinMaxScaler(),['Fare','FamilyMembers'])
    ],
    remainder='passthrough',
    verbose_feature_names_out=False
)

encoder_scaler.set_output(transform='pandas');

encoder_scaler.fit(x_train)
x_train = encoder_scaler.transform(x_train);
x_test = encoder_scaler.transform(x_test);

In [63]:
x_train.drop(columns='Cabin',inplace=True)
x_test.drop(columns='Cabin',inplace=True)

In [65]:
x_test.sample(3)

,Pclass,Sex_male,Embarked_Q,Embarked_S,Cabin_deck_B,Cabin_deck_C,Cabin_deck_D,Cabin_deck_E,Cabin_deck_F,Cabin_deck_G,Cabin_deck_M,Cabin_deck_T,Age,Fare,FamilyMembers,missingindicator_Cabin
25,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.704565,0.487668,0.6,True
835,2.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.784556,1.292030,0.2,False
538,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.024552,0.225286,0.0,True
